# 4장 2강: 질문·응답 로그 테이블 구현

## 2. queries 테이블 생성
```
drop table documents cascade;
drop table queries cascade;

create table documents (
  id bigint generated always as identity primary key,
  user_id uuid references auth.users (id),
  title text not null,
  content text,
  status text default 'draft' check (status in ('draft', 'published', 'archived')),
  created_at timestamptz default now()
);

create table queries (
  id bigint generated always as identity primary key,
  user_id uuid references auth.users (id),
  document_id bigint references documents (id),
  question text not null,
  status text default 'pending' check (status in ('pending', 'completed', 'failed')),
  created_at timestamptz default now()
);
```

## query_logs 테이블 생성
```
drop table query_logs cascade;

create table query_logs (
  id bigint generated always as identity primary key,
  query_id bigint references queries (id),
  answer text,
  model_name text,
  status text default 'success' check (status in ('success', 'error')),
  error_message text,
  created_at timestamptz default now()
);
```

# 4장 3강: SDK 기반 질문·응답 로그 저장

## 1. Supabase SDK와 클라이언트 준비

In [ ]:


# .env 파일의 값을 환경변수로 불러오기


# Client 초기화


## 2. 질문 저장 함수 create_query()

## 3. 응답 로그 저장 함수 save_query_log()

## 4. 실습: 질문·응답 저장과 이력 조회
### 1단계. 질문 저장하기

### 2단계. 더미 LLM 응답을 로그로 저장하기

### 3단계. 사용자별 질문 이력 조회 함수 작성

### 4단계. 오류 응답 처리하기


# 4장 4강: 트랜잭션이 필요한 AI 데이터 처리 시나리오
## 4. 실습: SQL 트랜잭션부터 Supabase RPC 구현까지
### 4단계. 파이썬 SDK에서 RPC 함수 호출 및 검증

#### Supabase SQL Editor에서 RPC 함수 정의하기
```
create or replace function ask_and_log(
  p_user_id uuid,
  p_document_id bigint,
  p_question text,
  p_answer text,
  p_model_name text
)
returns bigint -- 생성된 query의 ID를 반환
language plpgsql
as $$
declare
  v_query_id bigint;
begin
  -- queries 테이블에 질문 저장 및 생성된 ID 추출
  insert into queries (user_id, document_id, question, status)
  values (p_user_id, p_document_id, p_question, 'completed')
  returning id into v_query_id;

  -- query_logs 테이블에 응답 로그 저장 (v_query_id 활용)
  insert into query_logs (query_id, answer, model_name, status)
  values (v_query_id, p_answer, p_model_name, 'success');

  -- 성공 시 생성된 query_id 반환
  return v_query_id;

-- PL/pgSQL 함수 실행 도중 오류가 발생하면 전체 과정이 자동으로 ROLLBACK 됩니다.
end;
$$;
```

# 4장 7강: 통합 프로젝트 2 - Supabase SDK 기능 통합

## 1. SDK CRUD 통합과 Auth 연동

## 2. 문서 관리 기능 구현

## 3. 질문 로그 저장과 사용자별 조회

## 4. 실습: 기능 통합과 실행 검증

### 1단계. 문서 생성·조회·수정·삭제 이어서 실행

### 2단계. 질문 저장과 더미 응답 로그 저장

### 3단계. 사용자별 질문 조회 확인